In [1]:
import numpy as np
import scipy
import time
import scipy.sparse.linalg as spla

In [11]:
# function to build the tridiagonal matrix of size n x n
def build_matrix(n):
    row_indices = np.append(np.append(range(n), range(n-1)), range(1, n))
    col_indices = np.append(np.append(range(n), range(1, n)), range(n-1))
    data = np.append(np.append(2 * np.ones(n), -1 * np.ones(n-1)), -1 * np.ones(n-1))
    #print(data)
    matrix = scipy.sparse.csr_matrix((data, (row_indices, col_indices)), shape=(n, n))
    return matrix

In [13]:

# define the size of the system of equations
n = 500

# build matrix K and right hand side vector f
Ksparse = build_matrix(n)
Kdense = Ksparse.todense()
f = np.zeros(n)
f[int(n/2)] = 1

#print(Ksparse)
print(Kdense)

[[ 2. -1.  0. ...  0.  0.  0.]
 [-1.  2. -1. ...  0.  0.  0.]
 [ 0. -1.  2. ...  0.  0.  0.]
 ...
 [ 0.  0.  0. ...  2. -1.  0.]
 [ 0.  0.  0. ... -1.  2. -1.]
 [ 0.  0.  0. ...  0. -1.  2.]]


In [8]:
# build matrix K and right hand side vector f
Ksparse = build_matrix(n)
Kdense = Ksparse.todense()
f = np.zeros(n)
f[int(n/2)] = 1

# solve using different methods and time them
start_time = time.perf_counter()
u = scipy.linalg.inv(Kdense) @ f
t_inv = time.perf_counter() - start_time

start_time = time.perf_counter()
u = scipy.linalg.solve(Kdense, f)
t_scipy_solve = time.perf_counter() - start_time

start_time = time.perf_counter()
u = spla.spsolve(Ksparse, f)
t_spsolve = time.perf_counter() - start_time

Kcsc = build_matrix(n).tocsc()
start_time = time.perf_counter()
ilu = spla.spilu(Kcsc)  # factorization
M2 = spla.LinearOperator(Kcsc.shape, matvec=ilu.solve)
u = spla.cg(Ksparse, f, M=M2)
t_cg = time.perf_counter() - start_time

print(f"Time taken for matrix inversion and multiplication: {t_inv:.4f} seconds")
print(f"Time taken for scipy.linalg.solve: {t_scipy_solve:.4f} seconds")
print(f"Time taken for scipy.sparse.linalg.spsolve: {t_spsolve:.4f} seconds")
print(f"Time taken for scipy.sparse.linalg.cg with ILU preconditioner: {t_cg:.4f} seconds")

[ 2.  2.  2. ... -1. -1. -1.]
[ 2.  2.  2. ... -1. -1. -1.]
Time taken for matrix inversion and multiplication: 0.3483 seconds
Time taken for scipy.linalg.solve: 0.0144 seconds
Time taken for scipy.sparse.linalg.spsolve: 0.0072 seconds
Time taken for scipy.sparse.linalg.cg with ILU preconditioner: 0.0111 seconds


In [9]:
import scipy.linalg as la
import numpy as np


sigma = np.array([[2,4,-1],[4,5,-3],[-1,-3,5]])


In [10]:
lamb, N = la.eig(sigma)
print(f"Eigenvalues: {lamb}")
print(f"Eigenvectors: {N}")

Eigenvalues: [ 9.74487333+0.j -0.93334007+0.j  3.18846674+0.j]
Eigenvectors: [[-0.4393462  -0.77160499  0.46000072]
 [-0.71457255  0.6105039   0.34157146]
 [ 0.54439048  0.17863576  0.81959031]]


In [6]:
I1 = np.trace(sigma)
I2 = 0.5 * (I1**2 - np.trace(sigma @ sigma))
I3 = np.linalg.det(sigma)

print(f"First invariant: {I1}")
print(f"Second invariant: {I2}")
print(f"Third invariant: {I3}")

First invariant: 12
Second invariant: 19.0
Third invariant: -28.99999999999999
